# 19A1B — Corrected Temporal AIA Dataset / Loader Preparation

## Purpose

This notebook supersedes the first 19A1 preparation notebook.

The correction is methodological: **the frozen cross-cycle role-retention decision is applied first**, and only then are temporal AIA objects joined.

This prevents the AIA branch from silently re-admitting samples that were already excluded by the frozen protocol.

## Locked order of operations

1. Load the frozen role-candidate table.
2. Retain only rows with:
   - `structural_candidate == 1`
   - empty `exclusion_reasons`
3. Preserve the assigned role and authoritative label.
4. Join only the exact three historical AIA frames:
   - t−288 min
   - t−192 min
   - t−96 min
5. Verify image objects and stored channel order.
6. Save loader-ready manifests.
7. Do **not** train, calibrate, threshold, or evaluate any model.

## Scientific contract

- Target: same-active-region M/X flare occurrence in `(t, t+48 h]`
- AIA channels: 94, 131, 171, 193, 211, 335 Å
- Source image tensor expected only after verification: `(512, 512, 6)`
- Source dtype expected only after verification: `float32`
- Authoritative target: `original_label_48h_final` / exported as `label_48h_final`
- Embedded NPZ `y` is not authoritative
- Cycle-25 remains untouched for development
- Primary independent test: 2021–2025
- 2026: supplementary only

## Supervisor-methodology alignment

The image branch is prepared for CNN/ViT spatial encoders plus explicit temporal modelling, with later AIA+SHARP fusion. GOES remains label lineage rather than an input feature.


## Channel-label correction

The NPZ stores the verified six channels using labels
`aia94, aia131, aia171, aia193, aia211, aia335`.
These map directly and in order to the physical wavelengths
94, 131, 171, 193, 211, 335 Å.


In [1]:
from pathlib import Path
import gzip, json, subprocess, tempfile
from collections import Counter

import numpy as np
import pandas as pd

HOME = Path.home()
META = HOME / "aia17_metadata_stage1"
OUT = HOME / "aia19_temporal_aia_input_corrected"
OUT.mkdir(parents=True, exist_ok=True)

ROLE_PATH = META / "broad_cycle24_finalfit_v2/reports/20260916T191645934223Z/broad_cycle24_final_role_candidates.csv.gz"
TEMPORAL_PATH = META / "temporal_manifest_v1/reports/20260916T143622218859Z/temporal_sequence_candidates.jsonl.gz"

CHANNELS_EXPECTED = ["aia94", "aia131", "aia171", "aia193", "aia211", "aia335"]
LAGS = [288, 192, 96]
PRIMARY_TEST_YEARS = [2021, 2022, 2023, 2024, 2025]

print("ROLE_PATH:", ROLE_PATH)
print("TEMPORAL_PATH:", TEMPORAL_PATH)
print("OUT:", OUT)

for p in [ROLE_PATH, TEMPORAL_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)


ROLE_PATH: /home/abmoses2000/aia17_metadata_stage1/broad_cycle24_finalfit_v2/reports/20260916T191645934223Z/broad_cycle24_final_role_candidates.csv.gz
TEMPORAL_PATH: /home/abmoses2000/aia17_metadata_stage1/temporal_manifest_v1/reports/20260916T143622218859Z/temporal_sequence_candidates.jsonl.gz
OUT: /home/abmoses2000/aia19_temporal_aia_input_corrected


## 1. Load and freeze the retained role population

In [2]:
roles = pd.read_csv(ROLE_PATH)

required = [
    "target_sample_id",
    "stored_year",
    "HARPNUM",
    "NOAA_AR_clean",
    "original_label_48h_final",
    "region_component_id",
    "proposed_final_role",
    "structural_candidate",
    "exclusion_reasons",
    "training_authorised",
]
missing = [c for c in required if c not in roles.columns]
if missing:
    raise RuntimeError(f"Missing role columns: {missing}")

roles["exclusion_reasons"] = roles["exclusion_reasons"].fillna("").astype(str)

retained = roles[
    roles["structural_candidate"].astype(int).eq(1)
    & roles["exclusion_reasons"].str.strip().eq("")
].copy()

print("RAW role rows:", len(roles))
print("RETAINED rows:", len(retained))
print("\nRetained support by role:")
print(
    retained.groupby("proposed_final_role")["original_label_48h_final"]
    .agg(["count", "sum"])
    .to_string()
)

# Hard scientific checks against the already-established frozen support.
test_role = retained[retained["proposed_final_role"].eq("independent_cycle25_test")]
assert len(test_role) == 49329, len(test_role)
assert int(test_role["original_label_48h_final"].sum()) == 2351

supp_role = retained[retained["proposed_final_role"].eq("supplementary_2026")]
assert len(supp_role) == 2086, len(supp_role)
assert int(supp_role["original_label_48h_final"].sum()) == 135

dev_roles = {
    "cycle24_final_refit_pool",
    "cycle24_calibration_holdout",
    "cycle24_threshold_holdout",
}
dev_role_df = retained[retained["proposed_final_role"].isin(dev_roles)]
assert len(dev_role_df) == 64725, len(dev_role_df)

print("\nFrozen retained support checks passed.")


RAW role rows: 141644
RETAINED rows: 118057

Retained support by role:
                             count   sum
proposed_final_role                     
cycle24_calibration_holdout   6253   133
cycle24_final_refit_pool     52155  1759
cycle24_threshold_holdout     6317   202
cycle25_early_diagnostic      1917     0
independent_cycle25_test     49329  2351
supplementary_2026            2086   135

Frozen retained support checks passed.


## 2. Join exact three-frame temporal AIA histories to retained targets

In [3]:
retained_lookup = retained.set_index("target_sample_id").to_dict("index")
records = []
status_counts = Counter()

with gzip.open(TEMPORAL_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        sid = rec["target_sample_id"]
        if sid not in retained_lookup:
            continue

        r = retained_lookup[sid]
        frames = rec.get("frames", [])

        history_ok = (
            rec.get("history_status") == "NOMINAL_HISTORY_OBJECTS_AVAILABLE_TIMING_PENDING"
            and len(frames) == 3
            and [fr.get("lag_minutes") for fr in frames] == LAGS
            and all(fr.get("nominal_candidate") is True for fr in frames)
            and all(fr.get("object_status") == "EXACT_NONEMPTY_OBJECT" for fr in frames)
        )
        status_counts["history_ok" if history_ok else "history_not_ok"] += 1
        if not history_ok:
            continue

        uris = [fr.get("object_uri") for fr in frames]
        if not all(isinstance(u, str) and u.startswith("gs://") for u in uris):
            continue

        records.append({
            "target_sample_id": sid,
            "stored_year": int(r["stored_year"]),
            "HARPNUM": int(r["HARPNUM"]),
            "NOAA_AR_clean": int(r["NOAA_AR_clean"]),
            "region_component_id": r["region_component_id"],
            "label_48h_final": int(r["original_label_48h_final"]),
            "role": r["proposed_final_role"],
            "history_uri_tminus288": uris[0],
            "history_uri_tminus192": uris[1],
            "history_uri_tminus96": uris[2],
            "issue_utc": rec.get("issue_utc"),
            "issue_raw_TAI": rec.get("issue_raw_TAI"),
            "training_authorised_source": bool(rec.get("training_authorised", False)),
            "label_boundary_review_required": bool(rec.get("label_boundary_review_required", False)),
        })

manifest = pd.DataFrame(records)
if manifest.empty:
    raise RuntimeError("No retained temporal-AIA records found.")

print("Loader-ready retained records:", len(manifest))
print("Positives:", int(manifest["label_48h_final"].sum()))
print("Region components:", manifest["region_component_id"].nunique())

print("\nSupport by role:")
print(
    manifest.groupby("role")["label_48h_final"]
    .agg(["count", "sum"])
    .to_string()
)

# Because NOMINAL_HISTORY_INCOMPLETE_OR_EXCLUDED was already part of the frozen
# structural exclusions, the retained population should survive this join exactly.
expected_ids = set(retained["target_sample_id"])
actual_ids = set(manifest["target_sample_id"])

missing_after_join = expected_ids - actual_ids
unexpected_after_join = actual_ids - expected_ids

print("\nMissing retained IDs after AIA join:", len(missing_after_join))
print("Unexpected IDs after AIA join:", len(unexpected_after_join))

if missing_after_join or unexpected_after_join:
    raise RuntimeError("Temporal AIA join changed the frozen retained population.")


Loader-ready retained records: 118057
Positives: 4580
Region components: 2302

Support by role:
                             count   sum
role                                    
cycle24_calibration_holdout   6253   133
cycle24_final_refit_pool     52155  1759
cycle24_threshold_holdout     6317   202
cycle25_early_diagnostic      1917     0
independent_cycle25_test     49329  2351
supplementary_2026            2086   135

Missing retained IDs after AIA join: 0
Unexpected IDs after AIA join: 0


## 3. Build development, independent-test and supplementary manifests

In [4]:
development = manifest[manifest["role"].isin(dev_roles)].copy()

independent_test = manifest[
    manifest["role"].eq("independent_cycle25_test")
    & manifest["stored_year"].isin(PRIMARY_TEST_YEARS)
].copy()

supplementary = manifest[
    manifest["role"].eq("supplementary_2026")
    & manifest["stored_year"].eq(2026)
].copy()

assert len(development) == 64725
assert len(independent_test) == 49329
assert int(independent_test["label_48h_final"].sum()) == 2351
assert len(supplementary) == 2086
assert int(supplementary["label_48h_final"].sum()) == 135

dev_regions = set(development["region_component_id"])
test_regions = set(independent_test["region_component_id"])
overlap = dev_regions & test_regions
assert len(overlap) == 0

print("Cycle-24 development:", len(development), "positives:", int(development["label_48h_final"].sum()))
print("Cycle-25 independent 2021-2025:", len(independent_test), "positives:", int(independent_test["label_48h_final"].sum()))
print("Supplementary 2026:", len(supplementary), "positives:", int(supplementary["label_48h_final"].sum()))
print("Development/test region overlap:", len(overlap))

print("\nIndependent-test support by year:")
print(
    independent_test.groupby("stored_year")["label_48h_final"]
    .agg(["count", "sum"])
    .to_string()
)


Cycle-24 development: 64725 positives: 2094
Cycle-25 independent 2021-2025: 49329 positives: 2351
Supplementary 2026: 2086 positives: 135
Development/test region overlap: 0

Independent-test support by year:
             count   sum
stored_year             
2021          5104   126
2022          9999   119
2023         12392   278
2024         12553  1316
2025          9281   512


## 4. Verify the real stored AIA tensor and channel order

In [5]:
def gcloud_cp(uri: str, destination: Path):
    subprocess.run(
        ["gcloud", "storage", "cp", uri, str(destination)],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

# Pick one retained Cycle-24 sample and inspect all 3 frames.
row = development.iloc[0]
canary = []

with tempfile.TemporaryDirectory() as td:
    td = Path(td)

    for i, col in enumerate([
        "history_uri_tminus288",
        "history_uri_tminus192",
        "history_uri_tminus96",
    ]):
        uri = row[col]
        local = td / f"canary_{i}.npz"
        gcloud_cp(uri, local)

        with np.load(local, allow_pickle=False) as z:
            if "x" not in z or "channels" not in z:
                raise RuntimeError(f"Required image keys absent in {uri}")

            x = z["x"]
            channels = [str(v) for v in z["channels"].tolist()]

            rec = {
                "uri": uri,
                "x_shape": list(x.shape),
                "x_dtype": str(x.dtype),
                "channels": channels,
                "finite": bool(np.isfinite(x).all()),
                "embedded_y_present": "y" in z,
                "embedded_y_used_as_target": False,
            }
            canary.append(rec)

            assert x.shape == (512, 512, 6), x.shape
            assert str(x.dtype) == "float32", x.dtype
            assert channels == CHANNELS_EXPECTED, channels
            assert np.isfinite(x).all()

print(json.dumps(canary, indent=2))
print("\nAIA tensor/channel verification passed.")


[
  {
    "uri": "gs://suryabench-sharp-pipeline-bamidele/samples_npz/2010/20100521_1636_HARP26_NOAA11072.npz",
    "x_shape": [
      512,
      512,
      6
    ],
    "x_dtype": "float32",
    "channels": [
      "aia94",
      "aia131",
      "aia171",
      "aia193",
      "aia211",
      "aia335"
    ],
    "finite": true,
    "embedded_y_present": true,
    "embedded_y_used_as_target": false
  },
  {
    "uri": "gs://suryabench-sharp-pipeline-bamidele/samples_npz/2010/20100521_1812_HARP26_NOAA11072.npz",
    "x_shape": [
      512,
      512,
      6
    ],
    "x_dtype": "float32",
    "channels": [
      "aia94",
      "aia131",
      "aia171",
      "aia193",
      "aia211",
      "aia335"
    ],
    "finite": true,
    "embedded_y_present": true,
    "embedded_y_used_as_target": false
  },
  {
    "uri": "gs://suryabench-sharp-pipeline-bamidele/samples_npz/2010/20100521_1948_HARP26_NOAA11072.npz",
    "x_shape": [
      512,
      512,
      6
    ],
    "x_dtype": "float32"

## 5. Save corrected loader manifests and protocol

In [6]:
manifest.to_csv(OUT / "temporal_aia_loader_manifest.csv.gz", index=False, compression="gzip")
development.to_csv(OUT / "cycle24_aia_development_manifest.csv.gz", index=False, compression="gzip")
independent_test.to_csv(OUT / "cycle25_aia_independent_test_manifest.csv.gz", index=False, compression="gzip")
supplementary.to_csv(OUT / "aia_2026_supplementary_manifest.csv.gz", index=False, compression="gzip")

support = {
    "all_retained_loader_ready": {
        "n": int(len(manifest)),
        "positives": int(manifest["label_48h_final"].sum()),
        "regions": int(manifest["region_component_id"].nunique()),
    },
    "cycle24_development": {
        "n": int(len(development)),
        "positives": int(development["label_48h_final"].sum()),
        "regions": int(development["region_component_id"].nunique()),
    },
    "cycle25_2021_2025_independent": {
        "n": int(len(independent_test)),
        "positives": int(independent_test["label_48h_final"].sum()),
        "regions": int(independent_test["region_component_id"].nunique()),
    },
    "supplementary_2026": {
        "n": int(len(supplementary)),
        "positives": int(supplementary["label_48h_final"].sum()),
        "regions": int(supplementary["region_component_id"].nunique()),
    },
}

protocol = {
    "status": "CORRECTED_TEMPORAL_AIA_LOADER_PREPARED_CHANNEL_LABELS_VERIFIED_NO_MODEL_TRAINING",
    "supersedes": "19A1B_Temporal_AIA_dataset_loader_corrected",
    "correction": "Frozen structural-retention rules are applied before temporal AIA joining; stored NPZ channel labels are verified with their aia-prefix convention.",
    "forecast_target": "same-active-region M/X occurrence in (t,t+48h]",
    "channels_angstrom_verified": [94, 131, 171, 193, 211, 335],
    "stored_channel_order_verified": CHANNELS_EXPECTED,
    "physical_wavelength_order_angstrom": [94, 131, 171, 193, 211, 335],
    "verified_image_key": "x",
    "verified_image_shape": [512, 512, 6],
    "verified_image_dtype": "float32",
    "history_lags_minutes_oldest_to_newest": LAGS,
    "authoritative_label_column": "label_48h_final",
    "embedded_npz_y_authoritative": False,
    "cycle25_used_for_model_development": False,
    "primary_independent_test_years": PRIMARY_TEST_YEARS,
    "supplementary_2026_only": True,
    "region_overlap_cycle24_cycle25": int(len(overlap)),
    "support": support,
    "canary": canary,
    "model_trained": False,
    "calibration_fitted": False,
    "threshold_selected": False,
    "scientific_clearance": False,
    "notes": [
        "Supervisor methodology is implemented as model-design/evaluation requirements without publishing confidential correspondence.",
        "Frozen cross-cycle structural exclusions are inherited exactly.",
        "No embedded NPZ label is used as the authoritative target.",
        "Historical-source availability and label-clearance limitations remain inherited from earlier audit stages."
    ],
}

(OUT / "support_summary.json").write_text(json.dumps(support, indent=2) + "\n")
(OUT / "protocol_record.json").write_text(json.dumps(protocol, indent=2) + "\n")

print(json.dumps(support, indent=2))
print("OUTPUT:", OUT)
print("STATUS: CORRECTED_TEMPORAL_AIA_LOADER_PREPARED_CHANNEL_LABELS_VERIFIED_NO_MODEL_TRAINING")


{
  "all_retained_loader_ready": {
    "n": 118057,
    "positives": 4580,
    "regions": 2302
  },
  "cycle24_development": {
    "n": 64725,
    "positives": 2094,
    "regions": 1232
  },
  "cycle25_2021_2025_independent": {
    "n": 49329,
    "positives": 2351,
    "regions": 985
  },
  "supplementary_2026": {
    "n": 2086,
    "positives": 135,
    "regions": 49
  }
}
OUTPUT: /home/abmoses2000/aia19_temporal_aia_input_corrected
STATUS: CORRECTED_TEMPORAL_AIA_LOADER_PREPARED_CHANNEL_LABELS_VERIFIED_NO_MODEL_TRAINING


## 6. Handoff to 19A2

GPU training may begin only if this notebook completes with all hard assertions passing.

The next stage should:

1. keep Cycle-25 completely untouched;
2. stage/deduplicate AIA source objects efficiently;
3. train the AIA image branch on Cycle-24 development only;
4. use CNN and/or ViT spatial encoding with explicit temporal modelling;
5. handle class imbalance without using Cycle-25;
6. freeze calibration and threshold before any Cycle-25 evaluation;
7. preserve clean/executed notebooks, predictions, protocol, hashes and environment.
